# 03 - Train Random Forest Models

Train the 20 Random Forest regressors from the processed artefacts. This notebook writes only RF outputs and leaves the shared MLP and XGBoost artefacts untouched.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

project_root = '/content/drive/MyDrive/Dissertation_ML_Project_V1'
if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)
print(f'Project root set to: {project_root}')
print('Contents:', os.listdir('.'))


In [ ]:
from src.train_rf import train_all_rf

train_all_rf("data/processed", "models/RF", "tables")

In [ ]:
import json
import pandas as pd

with open("models/RF/rf_results.json") as f:
    results = json.load(f)

rows = []
for target, r in results.items():
    if "error" in r:
        rows.append({"target": target, "status": "ERROR"})
        continue
    rows.append({
        "target": target,
        "r2_cv": r.get("r2_cv"),
        "r2_test": r.get("r2_test"),
        "r2_train": r.get("r2_train"),
        "rmse": r.get("rmse"),
        "mae": r.get("mae"),
        "MU": r.get("max_underestimate"),
        "status": "OK",
        "time_min": r.get("train_time_seconds", 0) / 60,
    })

df = pd.DataFrame(rows)
print(f"Total models: {len(df)}")
print(f"R² >= 0.96: {(df['r2_test'] >= 0.96).sum()}/20")
print(f"Overfitting (train-test > 0.03): {((df['r2_train'] - df['r2_test']) > 0.03).sum()}/20")
df